## Homework 2: Vector Search

This is the personal implementation of the homework 1 of [LLM Zoomcamp 2026 Cohort](https://github.com/DataTalksClub/llm-zoomcamp/tree/main).This homework requires turning text into vectors, then searching by similarity. It also practices how to combine vector search with keyword search. The course repository is organized by modules and each module is a top-level folder with a `lessons/` subfolder of numbered markdown pages:

```python
01-agentic-rag/
└── lessons/
    ├── 01-intro.md
    ├── 02-environment.md
    ├── ...
    └── 16-other-frameworks.md
```

Each lesson page is a single markdown file, which are exactly the data to be fetched from GitHub and to be used as the knowledge base. Environment is prepared with:

```python
cd module2_homework
uv init --no-workspace
uv add onnxruntime tokenizers numpy tqdm minsearch gitsource
uv add --dev huggingface-hub jupyter

PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed
wget $PREFIX/download.py
wget $PREFIX/embedder.py

uv run python download.py
uv run python -m ipykernel install --user --name llm-zoomcamp-onnx --display-name "llm-zoomcamp-onnx"
```

After creating the new kernel, check if it is created properly:
```python
jupyter kernelspec list
```
Then if it appears in the list, but does not appear as Jupyter Kernel to change into, reload the window.


### Q1. Embedding a query

In [2]:
# pull the lesson pages from the course repository
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

print(f"Total lesson pages: {len(documents)}")

Total lesson pages: 72


In [3]:
from embedder import Embedder

embed = Embedder()

q = "How does approximate nearest neighbor search work?"
v = embed.encode(q)

print(f"The first value on embedded query: {v[0]:.2f}")

2026-06-26 21:10:57.758731307 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


The first value on embedded query: -0.02


### Q2. Cosine similarity

The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

Take the page `02-vector-search/lessons/07-sqlitesearch-vector.md`, embed its `content`, and compute the cosine similarity with the query vector from Q1. What do you get?

In [4]:
# find the page we need
target = "02-vector-search/lessons/07-sqlitesearch-vector.md"

result = next(
    (i for i, doc in enumerate(documents) if doc.get("filename") == target),
    None
)

print(f"Use page: {result}")
documents[result]

Use page: 22


{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [5]:
# get page content and embed
d  = documents[22]["content"]
dv = embed.encode(d)

# cosine similarity to earlier query
print(f"Similarity score: {v.dot(dv):.2f}")

Similarity score: 0.36


### Q3. Chunking and search by hand

A full page covers several topics, which waters down its embedding. Hence this part applies chunking. Chunking splits each page into smaller overlapping pieces and index those pieces instead.

In [6]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print(f"Total number of chunks: {len(chunks)}")
chunks[0]

Total number of chunks: 295


{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [7]:
import numpy as np

# get the content of chunks if exist
contents = [
    chunk["content"]
    for chunk in chunks
    if chunk.get("content")
]

# embed the contents
vectors = embed.encode_batch(contents)

# turn vectors into a matrix where rows are documents (vectors), and cols are dimensions of them
X = np.array(vectors)
print(f"There are {X.shape[0]} vectors with {X.shape[1]} dimensions.")

# score the previous query against all chunks
scores = X.dot(v)

There are 295 vectors with 384 dimensions.


In [8]:
# get the highest scored chunk, which is the most similar
idx = np.argmax(scores)
print(f"The most similar chunk index: {idx}, score: {scores[idx]}")

# get the filename of the most similar chunk
print(f"The filename of the most similar chunk: {chunks[idx]["filename"]}")


The most similar chunk index: 94, score: 0.6489016436447387
The filename of the most similar chunk: 02-vector-search/lessons/07-sqlitesearch-vector.md


### Q4. Vector search with minsearch
Use `VectorSearch` from `minsearch` and run a search for the following query: "What metric do we use to evaluate a search engine?"

In [9]:
from minsearch import VectorSearch

# index the chunks and vectors
vindex = VectorSearch()
vindex.fit(X, chunks)

In [10]:
# search for the given query
q2 = "What metric do we use to evaluate a search engine?"
v2 = embed.encode(q2)
results = vindex.search(v2, num_results=5)

# show the top result
print(f"File name of the top result: {results[0]["filename"]}")

File name of the top result: 04-evaluation/lessons/05-search-metrics.md


### Q5. Text search vs vector search

Vector search matches by meaning, keyword search by exact words. To compare them, index the same chunks with Index from minsearch. Use content as a text field. Run both searches for the query: "How do I store vectors in PostgreSQL?"

Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?

In [11]:
from minsearch import Index

# define a  function to index with minsearch
def build_index(documents):
    index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )
    index.fit(documents)
    return index

# create the chunks index
chunks_index = build_index(chunks)

In [23]:
# Perform a search with below query
q3 = "How do I store vectors in PostgreSQL?"

# get all related answers from all the courses (no filter, no boost, no numeric limit)
keyword_results = chunks_index.search(q3, num_results=5)

# compare results
print(f"File names appeared in keyword search:\n")
for item in keyword_results:
    print(item["filename"])

File names appeared in keyword search:

02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


In [24]:
# embed for vector search
v3 = embed.encode(q3)
vector_results = vindex.search(v3, num_results=5)

print(f"\nFile names appeared in vector search:\n")
for i in range(5):
    print(vector_results[i]["filename"])


File names appeared in vector search:

02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


The file which appears in vector search results but not in keyword search results: `02-vector-search/lessons/08-pgvector.md`

### Q6. Hybrid search

While vector search excels at matching semantic meaning, it often misses exact names or codes, whereas text search captures precise keywords but fails on synonyms. Hybrid search resolves these limitations by running both retrieval methods and merging their independent results. Since raw retrieval scores exist on completely different scales, Reciprocal Rank Fusion (RRF) combines them by looking strictly at a document's relative position (rank) within each list. It calculates a unified score by summing the reciprocal ranks across all lists using the formula:

$$RRF(d) = \sum_{m \in M} \frac{1}{k + \text{rank}_m(d)}$$

The smoothing constant $k$ (traditionally set to 60) dictates how heavily top-ranked results are weighted compared to cross-list consistency, successfully elevating documents that perform well in both retrieval strategies. A document that ranks well in both lists ends up higher than one that's only strong in a single list.

In [26]:
# rff for unified score
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

q4 = "How do I give the model access to tools?"

In [28]:
# text search results
txt_results = chunks_index.search(q4, num_results=5)

# vector search results
v4 = embed.encode(q4)
vec_results = vindex.search(v4, num_results=5)

# unified results
results = rrf([vec_results, txt_results])

# show the top results
print(f"File name of the top result in text results: {txt_results[0]["filename"]}")
print(f"File name of the top result in keyword results: {vec_results[0]["filename"]}")
print(f"File name of the top result in unified results: {results[0]["filename"]}")

File name of the top result in text results: 01-agentic-rag/lessons/14-agentic-loop.md
File name of the top result in keyword results: 01-agentic-rag/lessons/01-intro.md
File name of the top result in unified results: 01-agentic-rag/lessons/13-function-calling.md
